<!-- # Pipeline smoke test

Raw Citi Bike data -> `RawModelData` -> `ResolvedModelData` -> `Environment` -> `SimulationLog`.

The base scenario reproduces historical **demand** exactly. `FormDeparturesPhase`
re-releases every historical departure (gated by stock), and
`FormPotentialTripsPhase` assigns each departure a destination and duration from
the OD demand model `P(target | source, commodity)` + mean historical duration.

To make the replay exact, the resolved data is built with `saturate_stock=True`:
artificial saturated stock and dock capacities replace the GBFS snapshot, so the
demand gate and the overflow redirect stay in the pipeline but never bind. (The
GBFS snapshot is a current observation, unrelated to the historical start state,
and would starve the replay with stockouts that never happened.)

Because targets and durations come from the (aggregate) OD model rather than each
trip's own record, the per-trip journal is **not** identical to history, and the
OD-count matrix drifts slightly under per-period largest-remainder rounding. The
marginal that is preserved exactly is the departures table:
`simulated_departures_df == historical_departures_df`. -->

In [1]:
import pandas as pd

from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation
from gbp.consumers.simulator.engine import Environment, EnvironmentConfig
from gbp.consumers.simulator import (
    DockArrivals,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
)

In [2]:
# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path="/mnt/outer/Documents/vlzm/GFDRR_ubuntu/GFDRR/data/raw/202602-citibike-tripdata_1.csv",
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

# Graph (resolved) model data: period grid, historical flows, replay demand.
# saturate_stock=True swaps the GBFS snapshot for artificial saturated stock and
# capacities, so demand gating and overflow redirect stay in the pipeline but
# never bind -- the base replay reproduces the       historical departures exactly.
graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1), scale_capacity_factor = 10)

historical_flows_df = graph_data.historical_flows_df

In [3]:
# import numpy as np

# flows = graph_data.historical_flows_df  # .copy() не нужен — исходный фрейм ты не мутируешь

# departed = (
#     flows.loc[flows["event_type"] == "departed",
#               ["period_id", "source_id", "commodity_category", "quantity"]]
#     .groupby(["period_id", "source_id", "commodity_category"], as_index=False)["quantity"].sum()
#     .rename(columns={"source_id": "facility_id"})
# )
# departed["quantity"] *= -1
# departed["event_type"] = "departed"

# arrived = (
#     flows.loc[flows["event_type"] == "arrived",
#               ["period_id", "realized_target_id", "commodity_category", "quantity"]]
#     .groupby(["period_id", "realized_target_id", "commodity_category"], as_index=False)["quantity"].sum()
#     .rename(columns={"realized_target_id": "facility_id"})
# )
# arrived["event_type"] = "arrived"

# total = pd.concat([departed, arrived], ignore_index=True)

# facilities  = np.sort(graph_data.facilities_df["facility_id"].unique())
# commodities = np.sort(graph_data.commodities_categories_df["commodity_category"].unique())
# periods     = np.sort(graph_data.periods_df["period_id"].unique())
# event_types = ["departed", "arrived"]  # departed раньше arrived

# full_index = pd.MultiIndex.from_product(
#     [facilities, commodities, periods, event_types],
#     names=["facility_id", "commodity_category", "period_id", "event_type"],
# )

# total_flows_final = (
#     total.set_index(["facility_id", "commodity_category", "period_id", "event_type"])["quantity"]
#          .reindex(full_index, fill_value=0)
#          .reset_index()
# )

# keys = ["facility_id", "commodity_category"]
# total_flows_final["inventory"] = total_flows_final.groupby(keys, sort=False)["quantity"].cumsum()
# total_flows_final["offset"] = (
#     -total_flows_final.groupby(keys, sort=False)["inventory"].transform("min")
# ).clip(lower=0)
# total_flows_final["inventory_feasible"] = total_flows_final["inventory"] + total_flows_final["offset"]
# total_flows_final["capacity"] = (
#     total_flows_final.groupby(keys, sort=False)["inventory_feasible"].transform("max")
# )

In [4]:
phases_canonical = [
    DockArrivals("previous"),
    FormDeparturesPhase(),
    FormPotentialTripsPhase(),
    DockArrivals("same"),
]

env_canonical = Environment(
    graph_data,
    EnvironmentConfig(phases=phases_canonical, 
                      seed=42, 
                      scenario_id="historical_replay", 
                      demand_scale_factor=1.0,
                      number_of_periods = 30),
)
env_canonical.run()

# Wire the finished run back into the graph-data container's simulated_* slots.
attach_simulation(graph_data, env_canonical.simulated_flows_df)
simulated_flows_df = graph_data.simulated_flows_df
simulated_departures_df = graph_data.simulated_departures_df
historical_departures_df = graph_data.historical_departures_df


def _sorted(df):
    return (df.sort_values(["period_id", "facility_id", "commodity_category"])
            .reset_index(drop=True))


# The base scenario reproduces historical demand exactly: the departures marginal
# of the simulated journal equals the historical one. (The per-trip journal is not
# identical, because targets/durations are drawn from the aggregate OD model.)
# pd.testing.assert_frame_equal(_sorted(simulated_departures_df), _sorted(historical_departures_df[historical_departures_df['period_id'].isin(simulated_flows_df['period_id'].unique())]))
# print("simulated_departures_df == historical_departures_df:",
#       _sorted(simulated_departures_df).equals(_sorted(_sorted(historical_departures_df[historical_departures_df['period_id'].isin(simulated_flows_df['period_id'].unique())]))))

In [5]:
simulated_flows_df

,event_id,period_id,flow_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,start_period,planned_end_period,realized_end_period,resource_id,quantity,reason
0,0,0,sim_0_0,user_trip,departed,classic_bike,5210.01,5411.08,<NA>,0,23,<NA>,<NA>,1,<NA>
1,1,3,sim_3_0,user_trip,departed,classic_bike,6575.03,6659.01,<NA>,3,23,<NA>,<NA>,1,<NA>
2,2,3,sim_3_1,user_trip,departed,classic_bike,7484.05,6659.01,<NA>,3,23,<NA>,<NA>,1,<NA>
3,3,6,sim_6_0,user_trip,departed,classic_bike,4550.05,4830.02,<NA>,6,25,<NA>,<NA>,1,<NA>
4,4,7,sim_7_0,user_trip,departed,classic_bike,4196.05,4425.02,<NA>,7,22,<NA>,<NA>,1,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16574,16574,29,sim_29_997,user_trip,departed,classic_bike,6560.01,6809.07,<NA>,29,30,<NA>,<NA>,1,<NA>
16575,16575,29,sim_29_998,user_trip,departed,electric_bike,6560.15,6756.01,<NA>,29,29,<NA>,<NA>,1,<NA>
16576,16576,29,sim_29_998,user_trip,arrived,electric_bike,6560.15,6756.01,6756.01,29,29,29,<NA>,1,<NA>
16577,16577,29,sim_29_999,user_trip,departed,classic_bike,6561.01,6727.02,<NA>,29,29,<NA>,<NA>,1,<NA>
